In [7]:
import pandas as pd
import regex as re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/siavashj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
df_orig = pd.read_parquet('FNDDSeverything.parquet.gzip')
df = df_orig.select_dtypes(include="str")

In [ ]:

def preprocess_food(text, Tokenize = True):
    """
    Complete preprocessing for food descriptions
    - Removes measurements and numbers
    - Tokenizes
    - Removes stopwords
    - Lemmatizes
    """

    sentence = ''
    
    # Step 0: replace ',' with ' '.
    text = re.sub(r',', ' ', text)
    
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove all measurement patterns
    patterns_to_remove = [
        r'\d+\.?\d*\s?(?:oz|lb|lbs|g|kg|ml|l|cup|cups|tsp|tbsp|pt|qt|gal|mg|mcg|cal|count|ct|pack|pk|piece|pieces|pc|ounce|ounces|pound|pounds|gram|grams|kilogram|kilograms|milliliter|milliliters|liter|liters|teaspoon|teaspoons|tablespoon|tablespoons|pint|pints|quart|quarts|gallon|gallons|milligram|milligrams|microgram|micrograms|calorie|calories|fl|fluid)\b',
        r'\b\d+\.?\d*\b',  # standalone numbers
        r'\d+\.?\d*\s?%',  # percentages
        r'\d+\s?x\s?\d+',  # dimensions
    ]
    
    for pattern in patterns_to_remove:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Step 3: Clean special characters but keep spaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 4: Clean multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 5: Tokenize
    tokens = word_tokenize(text)
    
    # Step 6: Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    
    # Step 7: Remove very short tokens
    tokens = [t for t in tokens if len(t) > 2]
    
    # Step 8: Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    if Tokenize:
        return tokens
    else:
        
        sentence = " ".join(tokens)
        return sentence

# Test

for sample in df.description[[24,81,124,134,550,551,552,553]]:
    tokens = preprocess_food(sample, True )
    print(f"Original: {sample}")
    print(f"Tokens:   {tokens}\n")
    sent = preprocess_food(sample, False )
    print(f"after token sent:   {sent}\n")



# def preprocess_column (List = list):
#     L = pd.DataFrame([])
#     for w in List:
#         preprocess_food_entries(w)
#     return L


Original:  DEATH WISH COFFEE CO LATTE, 8 FL OZ
Tokens:   ['death', 'wish', 'coffee', 'latte']

after token sent:   death wish coffee latte

Original:  ORIGINAL ICED W/ MILK CAPPUCCINO
Tokens:   ['original', 'iced', 'milk', 'cappuccino']

after token sent:   original iced milk cappuccino

Original:  ZERO CALORIE SPORTS DRINK, FRUIT PUNCH
Tokens:   ['zero', 'calorie', 'sport', 'drink', 'fruit', 'punch']

after token sent:   zero calorie sport drink fruit punch

Original: ""BACN,CAND,FL,4PC, Z""
Tokens:   ['bacn', 'cand']

after token sent:   bacn cand

Original: .38 SPECIAL GOOD ON EVERYTHING SEASONING, .38 SPECIAL
Tokens:   ['special', 'good', 'everything', 'seasoning', 'special']

after token sent:   special good everything seasoning special

Original: .5-.75 WGGS LANE SNAPPER
Tokens:   ['wggs', 'lane', 'snapper']

after token sent:   wggs lane snapper

Original: .5-1 LB IPW CLEAN OCTOPUS
Tokens:   ['ipw', 'clean', 'octopus']

after token sent:   ipw clean octopus

Original: .5/1 LB W.